# Activity 12: Animating data with Plotly

DSC 402/502 · Data Visualization · Prof. Steven Wilson

* **Part 1** is a worked example with the Gapminder data. Run each cell and try the changes it suggests.
* **Part 2** is your turn, with a different dataset.

## Part 1: Worked example

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

df = px.data.gapminder()
df.head()

,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4


The data has one row per country per year (1952 to 2007, every 5 years). That shape, one row per item per time step, is what Plotly needs for an animation.

First let's see what the entire thing looks like without animating, just plotting all of the data.

In [ ]:
fig = px.scatter(df, x="gdpPercap", y="lifeExp",
                 size="pop", color="continent", hover_name="country",
                 log_x=True, size_max=50)
fig.show()

### A first animation

`animation_frame` tells Plotly which column to step through. Run the cell, press the play button, and watch the axes

In [ ]:
fig = px.scatter(df, x="gdpPercap", y="lifeExp",
                 size="pop", color="continent", hover_name="country",
                 log_x=True, size_max=50,
                 animation_frame="year")
fig.show()

Two problems:

1. **The axes only fit the first year.** Plotly sizes the axes for 1952, so countries that grow later can drift off the edges.
2. **Plotly does not know which dot is which country.** Without that, it can't be sure to move each country's dot to that same country's next position. In this example it happens to work because the data was already sorted by contry first, then row, but it's better to be safe...

### Fixed axes and object constancy

* `range_x` and `range_y` fix the axes for the whole animation. Fixed axes also give viewers a stable frame of reference, so movement on screen always means a change in the data.
* `animation_group` names the column that identifies each mark across frames, so each dot stays the same country.

In [ ]:
fig = px.scatter(df, x="gdpPercap", y="lifeExp",
                 size="pop", color="continent", hover_name="country",
                 log_x=True, size_max=50,
                 animation_frame="year", animation_group="country",
                 range_x=[100, 150000], range_y=[20, 90])
fig.show()

### Timing settings

Plotly handles its timing settings inside the play button. The function below sets them to make it simpler, but it's worth looking at briefly.

* **frame_ms**: how long each year stays on screen, in milliseconds.
* **transition_ms**: how long the movement between years takes. Keep it at or below `frame_ms`, or the next frame starts before the movement finishes.
* **easing**: how the speed changes during the movement. `"linear"` is constant speed. `"cubic-in-out"` starts slow, speeds up, and slows down at the end. Other options include `"quad-in-out"`, `"elastic"` and `"bounce"`.

Plotly has no loop setting; the animation plays once each time you press play.

In [ ]:
def set_timing(fig, frame_ms=800, transition_ms=500, easing="cubic-in-out"):
    play = fig.layout.updatemenus[0].buttons[0].args[1]
    play["frame"]["duration"] = frame_ms
    play["transition"]["duration"] = transition_ms
    play["transition"]["easing"] = easing
    fig.layout.sliders[0].transition = {"duration": transition_ms, "easing": easing}
    return fig

In [ ]:
set_timing(fig, frame_ms=900, transition_ms=500, easing="cubic-in-out")
fig.show()

Try a few combinations and notice what changes:

* `frame_ms=300, transition_ms=300` (fast)
* `frame_ms=2000, transition_ms=1500` (slow)
* `easing="linear"` vs `easing="cubic-in-out"`
* `transition_ms=0` (no movement, the dots jump)

### Highlighting one item

People can only track a few moving objects at once. If one item matters to your story, make it stand out.

In [ ]:
focus = "China"
df["highlight"] = np.where(df["country"] == focus, focus, "Other countries")

fig = px.scatter(df, x="gdpPercap", y="lifeExp",
                 size="pop", color="highlight", hover_name="country",
                 color_discrete_map={focus: "#d62728", "Other countries": "lightgray"},
                 log_x=True, size_max=50,
                 animation_frame="year", animation_group="country",
                 range_x=[100, 150000], range_y=[20, 90])
set_timing(fig, frame_ms=1000, transition_ms=800)
fig.show()

### Static alternatives

Animation is not always the best choice. Here are two static views of the same data for a handful of countries.

**Trails**: each country's full path at once.

In [ ]:
countries = ["Zimbabwe", "Botswana", "Rwanda", "South Africa", "China", "India",
             "Korea, Rep.", "United States", "Japan", "Brazil", "Iraq", "Cambodia"]
some = df[df["country"].isin(countries)]

fig = px.line(some, x="gdpPercap", y="lifeExp", color="country",
              markers=True, log_x=True, hover_data=["year"])
fig.show()

**Small multiples**: one small chart per country, all on the same scale. Uses the concept of a `facet`, which is which dimension we break down the multiple charts by.

In [ ]:
fig = px.line(some, x="year", y="lifeExp",
              facet_col="country", facet_col_wrap=4, height=600)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

### Recording an animation

To share an animation, screen-record it while it plays:

* **Mac:** press Cmd+Shift+5, choose **Record Selected Portion**, and draw a box around the chart.
* **Windows:** open the Snipping Tool, switch to video, and draw a box around the chart. (Win+Alt+R in Xbox Game Bar also works.)

Press play on the chart, then stop the recording when the animation ends. The video file can be posted straight to Discord.

## Part 2: Your turn

1. Pick a dataset that changes over time and is not Gapminder. One good source is [Our World in Data](https://ourworldindata.org/):
    * open any chart, choose **Download**, and get the CSV. These files usually have Entity, Code and Year columns plus one value column. Make sure there is a time column or something you can animate over.
2. Load it and get it into long form: one row per item per time step, if it is not already.
3. Make an animated chart with `animation_frame` and `animation_group`.
4. Choose your settings: axis ranges, `frame_ms`, `transition_ms`, easing, and whether to highlight one item.
5. Screen-record your animation playing (see "Recording an animation" in Part 1).
6. Post in **#12-animation**:
    * your screen recording, data source, and what you think the animation does better than a static visualization

In [33]:
# Load your data here

# Data Source for Air Pollution Mortality Rate: https://www.stateofglobalair.org/data/#/health/plot?country=AFG&pollutant=pm25&measure=death&deathMetric=rate&geography=country&region=country&subregions=&outcome=burden&regionToggle=0&globals=false&hideCountry=false
# Data Source for Population: https://data.worldbank.org/indicator/SP.POP.TOTL
# Data Source for Life Expectancy: https://ourworldindata.org/grapher/life-expectancy

# Load Pollution Dataset
pollution_df = pd.read_excel("/content/Ambient_Particle_Air_Pollution_Death_Rates.xlsx")

# Melt Pollution Dataset for each row to have a Country and Year.
pollution_df = pollution_df.melt(
    id_vars=["Year"],
    var_name="Country",
    value_name="Death Rate per 100k"
)

# Load Population Dataset
population_df = pd.read_excel("/content/Countries_Populations.xlsx")

# Melt Population Dataset for each row to have a Country and Year.
population_df = population_df.melt(
    id_vars=["Country", "Continent", "Code"],
    var_name="Year",
    value_name="Population"
)

# Life Expectancy Dataset did not need to be melted.
life_expectancy_df = pd.read_excel("/content/Life_Expectancy.xlsx")

# I pre-processed the datasets to have the exact same countries and years. Now merge them.
pollution_df["Year"] = pollution_df["Year"].astype(int)
population_df["Year"] = population_df["Year"].astype(int)
mortality_df = pd.merge(pollution_df, population_df, on=["Country", "Year"], how="inner")
mortality_df = pd.merge(mortality_df, life_expectancy_df, on=["Country", "Year", "Code"], how="inner")

mortality_df

,Year,Country,Death Rate per 100k,Continent,Code,Population,Life Expectancy
0,1990,Afghanistan,47.7,Asia,AFG,12045660,45.1183
1,1991,Afghanistan,44.5,Asia,AFG,12238879,45.5207
2,1992,Afghanistan,41.7,Asia,AFG,13278974,46.5691
3,1993,Afghanistan,39.7,Asia,AFG,14943172,51.0212
4,1994,Afghanistan,37.6,Asia,AFG,16250794,50.9689
...,...,...,...,...,...,...,...
6761,2019,Zimbabwe,11.2,Africa,ZWE,15271368,61.0603
6762,2020,Zimbabwe,10.2,Africa,ZWE,15526888,61.5300
6763,2021,Zimbabwe,12.2,Africa,ZWE,15797210,60.1347
6764,2022,Zimbabwe,13.0,Africa,ZWE,16069056,62.3601


In [40]:
# Animated chart
# explore chart types and what plotly express can do: https://plotly.com/python/plotly-express/

fig = px.scatter(mortality_df, x="Death Rate per 100k", y="Life Expectancy",
                 size="Population", color="Continent", hover_name="Country",
                 title="Global Life Expectancy vs. Ambient Particle Air Pollution Death Rates (1990-2023)",
                 log_x=True, size_max=50,
                 animation_frame="Year", animation_group="Country",
                 range_x=[0.1, 250], range_y=[20, 90])
set_timing(fig, frame_ms=900, transition_ms=500, easing="cubic-in-out")
fig.show()